# E5 — Controlled audit: conventional CSV vs semantic RDF/SPARQL extraction

**Goal.** Assess which properties can be *demonstrated* from the same TalTech source observations using (A) the selected raw CSV and (B) the RDF/SPARQL-extracted observation table. We compare record preservation, semantic identifier availability, controlled integrity checks, and *file-loading/normalization* cost. **This is not a test of forecasting RMSE, nor a benchmark of complete RDF construction and query execution.**

**Validity:** input source and extraction are pre-existing artifacts from the same project. The RDF-export table has already been generated; this notebook does not measure RDF construction, SPARQL execution time, or full pipeline preparation efficiency. Errors are injected in *copies*, never in original data. To claim a special benefit of semantic constraints, compare fair CSV controls that are provided with equivalent schema information. Results are descriptive, not a causal trial of complete workflows.

Outputs: `E5_comparison_results/` with CSV and JSON files suitable for audit and manuscript decisions.

In [ ]:
from pathlib import Path
import zipfile, io, time, json, hashlib, platform
import numpy as np
import pandas as pd
from IPython.display import display

# Place the notebook beside the archived "outputs(1).zip", OR beside an "outputs/" directory.
# If stored elsewhere, replace with your absolute local path.
SOURCE = Path("outputs(1).zip")
OUTPUT_DIR = Path("E5_comparison_results")
OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
RAW = "outputs/01_extraction/02_selected_raw_observations_long.csv"
SEM = "outputs/04_sparql/03_ml_extraction_long_traceable.csv"

if not SOURCE.exists():
    if Path("outputs").is_dir(): SOURCE=Path("outputs")
    else: raise FileNotFoundError("Set SOURCE to outputs(1).zip or to the outputs/ directory")

def read_source(relative):
    if SOURCE.is_dir(): return pd.read_csv(SOURCE/relative.replace('outputs/','',1))
    with zipfile.ZipFile(SOURCE) as z:
        with z.open(relative) as f: return pd.read_csv(f)

def bytes_hash(relative):
    h=hashlib.sha256()
    if SOURCE.is_dir():
        with open(SOURCE/relative.replace('outputs/','',1),'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    else:
        with zipfile.ZipFile(SOURCE) as z:
            with z.open(relative) as f:
                for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

print("Source:",SOURCE.resolve())
print("Output:",OUTPUT_DIR.resolve())

## 1. Paired input inventory
Same 7 operational identifiers and three measured variables; raw extraction has source-file metadata, while the semantic/SPARQL export provides `point` and `observation` URIs. **The observation URI alone is not proof of complete lineage through resampling.**

In [ ]:
t0=time.perf_counter(); raw=read_source(RAW); raw_read_s=time.perf_counter()-t0
t0=time.perf_counter(); sem=read_source(SEM); sem_read_s=time.perf_counter()-t0
assert set(['device_id','timestamp','variable','value','unit']).issubset(raw)
assert set(['uuid','timestamp','variable','value','unit','point','observation']).issubset(sem)

print('Raw rows:',len(raw),' RDF-export rows:',len(sem))
print('Raw columns:',raw.columns.tolist())
print('RDF-export columns:',sem.columns.tolist())
print('Distinct semantic points:',sem.point.nunique(),' observation URIs:',sem.observation.nunique())
print('Raw-file read (s):',round(raw_read_s,3),' RDF-export CSV read (s):',round(sem_read_s,3))
assert len(raw)==len(sem)==323266, 'Source files differ from the E3-J audit: inspect before publication.' 

## 2. Exact key and approximate-value reconciliation
Join on original UTC timestamps + UUID + measured variable. Use one-to-one checks; compare values both bit/decimal-exactly and with a declared absolute tolerance of `1e-5` (to distinguish text serialization rounding from genuine value changes). This is **not** a comparison of independent acquisition methods: the RDF export was derived from these source measurements.

In [ ]:
t0=time.perf_counter()
a=raw.rename(columns={'device_id':'uuid'}).copy()
b=sem.copy()
for df in (a,b):
    df['timestamp']=pd.to_datetime(df['timestamp'],utc=True,errors='raise')
    df['value']=pd.to_numeric(df['value'],errors='raise')
key=['uuid','timestamp','variable']
assert not a.duplicated(key).any(), 'Duplicate raw composite key'
assert not b.duplicated(key).any(), 'Duplicate RDF composite key'
j=a[key+['value','unit']].merge(b[key+['value','unit']],on=key,how='outer',suffixes=('_csv','_rdf'),indicator=True,validate='one_to_one')
common=j['_merge'].eq('both')
delta=(j.loc[common,'value_csv']-j.loc[common,'value_rdf']).abs()
equal_unit=j.loc[common,'unit_csv'].eq(j.loc[common,'unit_rdf'])
reconcile={
 'n_raw':len(a),'n_rdf':len(b),'n_common_keys':int(common.sum()),
 'n_only_raw':int((j['_merge']=='left_only').sum()),
 'n_only_rdf':int((j['_merge']=='right_only').sum()),
 'n_exact_numeric_equal':int((delta==0).sum()),
 'n_numeric_differences_at_1e-9':int((delta>1e-9).sum()),
 'n_numeric_differences_at_1e-5':int((delta>1e-5).sum()),
 'max_abs_numeric_difference':float(delta.max()),
 'n_unit_mismatch':int((~equal_unit).sum()),
 'time_to_merge_and_compare_seconds':round(time.perf_counter()-t0,3)
}
display(pd.DataFrame([reconcile]).T.rename(columns={0:'value'}))
(OUTPUT_DIR/'01_record_reconciliation.json').write_text(json.dumps(reconcile,indent=2,default=str))
pd.DataFrame({'variable':j.loc[common,'variable'],'abs_delta':delta}).groupby('variable').agg(n=('abs_delta','size'),n_nonzero=('abs_delta',lambda s:int((s>1e-9).sum())),max_abs=('abs_delta','max')).to_csv(OUTPUT_DIR/'02_numeric_precision_by_variable.csv')
assert reconcile['n_common_keys']==len(raw) and reconcile['n_numeric_differences_at_1e-5']==0, 'Investigate unexpected input differences.' 

## 3. Metadata and identity audit
Two perspectives are reported separately: (i) whether fields are present in a given export, and (ii) whether they are actually verified by an external authoritative source. An RDF URI can be assigned without proving the underlying equipment-to-IFC mapping.

In [ ]:
metadata_rows=[]
for name,df,id_col in [('CSV raw',raw,'device_id'),('RDF/SPARQL export',sem,'uuid')]:
    metadata_rows.append({'pipeline':name,'rows':len(df),'devices':int(df[id_col].nunique()),'variables':int(df.variable.nunique()),
      'has_device_id':int(df[id_col].notna().sum()),'has_timestamp':int(df.timestamp.notna().sum()),'has_variable':int(df.variable.notna().sum()),
      'has_value':int(df.value.notna().sum()),'has_unit':int(df.unit.notna().sum()),
      'has_semantic_point_uri':int(df['point'].notna().sum()) if 'point' in df else 0,
      'has_observation_uri':int(df['observation'].notna().sum()) if 'observation' in df else 0,
      'has_source_file':int(df.source_file.notna().sum()) if 'source_file' in df else 0})
meta_table=pd.DataFrame(metadata_rows)
display(meta_table)
meta_table.to_csv(OUTPUT_DIR/'03_metadata_inventory.csv',index=False)

semantic_checks={
 'unique_observation_uris':bool(sem.observation.is_unique),
 'num_distinct_point_uris':int(sem.point.nunique()),
 'point_uris_per_uuid_variable_max':int(sem.groupby(['uuid','variable']).point.nunique().max()),
 'point_uris_per_uuid_variable_min':int(sem.groupby(['uuid','variable']).point.nunique().min()),
 'any_native_uuid_to_ifc_mapping_proven_by_these_tables':False,
 'provenance_through_resampling_proven_by_these_tables':False
}
print(semantic_checks)
(OUTPUT_DIR/'04_semantic_export_checks.json').write_text(json.dumps(semantic_checks,indent=2))

## 4. Controlled data-integrity perturbations — fair checks
Six synthetic fault types are injected into **one row at a time** and evaluated by: **(A)** conventional schema/metadata validation on the raw CSV; **(B)** a semantic-aware validator on the RDF-derived export. The CSV validator gets equivalent unit and variable dictionaries, so we do not artificially disadvantage it. The semantic-aware validator can additionally check `point` and `observation` URI fields unavailable in the raw CSV. For faults specific to semantic fields, the CSV check is **not applicable**, not a detection failure.

**Important:** These are Python-level validation checks on exported tables, **not SHACL execution or an RDF parser**. They cannot justify a claim that RDF inherently detects more errors than a well-specified conventional validation system.

In [ ]:
UNIT_EXPECTED={'temperature':'C','relative_humidity':'%','co2':'ppm'}
VARIABLES=set(UNIT_EXPECTED)
known_devices=set(raw.device_id)
known_points=set(sem.point)

def csv_validator(row):
    errors=[]
    if pd.isna(row.get('device_id')) or str(row.get('device_id')) not in known_devices:errors.append('invalid_device')
    if pd.isna(row.get('timestamp')):errors.append('missing_timestamp')
    var=row.get('variable')
    if var not in VARIABLES:errors.append('unknown_variable')
    elif row.get('unit')!=UNIT_EXPECTED[var]:errors.append('wrong_unit')
    if pd.isna(row.get('value')):errors.append('missing_value')
    return errors

def semantic_export_validator(row):
    errors=[]
    if pd.isna(row.get('uuid')) or str(row.get('uuid')) not in known_devices:errors.append('invalid_device')
    if pd.isna(row.get('timestamp')):errors.append('missing_timestamp')
    var=row.get('variable')
    if var not in VARIABLES:errors.append('unknown_variable')
    elif row.get('unit')!=UNIT_EXPECTED[var]:errors.append('wrong_unit')
    if pd.isna(row.get('value')):errors.append('missing_value')
    if pd.isna(row.get('point')) or row.get('point') not in known_points:errors.append('invalid_point_uri')
    if pd.isna(row.get('observation')):errors.append('missing_observation_uri')
    return errors

base_raw=raw.iloc[0].copy(); base_sem=sem.iloc[0].copy()
assert not csv_validator(base_raw) and not semantic_export_validator(base_sem)
FAULTS=[
  ('missing_device', 'device_id', 'uuid', None, True),
  ('missing_timestamp','timestamp','timestamp',None,True),
  ('invalid_variable','variable','variable','not_a_variable',True),
  ('wrong_unit','unit','unit','kg',True),
  ('missing_value','value','value',np.nan,True),
  ('invalid_point_uri',None,'point','https://invalid.example.org/point',False),
  ('missing_observation_uri',None,'observation',None,False),
]
rows=[]
for fault,r_col,s_col,bad,shared in FAULTS:
    r=base_raw.copy();s=base_sem.copy()
    if shared:r[r_col]=bad
    s[s_col]=bad
    a_result=csv_validator(r) if shared else None
    b_result=semantic_export_validator(s)
    rows.append({'fault':fault,'shared_field':shared,'CSV_applicable':shared,
                 'CSV_detected':bool(a_result) if shared else None,
                 'RDF_export_detected':bool(b_result),
                 'CSV_flags':'; '.join(a_result) if shared else 'NOT APPLICABLE',
                 'RDF_export_flags':'; '.join(b_result)})
fault_table=pd.DataFrame(rows)
display(fault_table)
fault_table.to_csv(OUTPUT_DIR/'05_controlled_fault_checks.csv',index=False)
assert fault_table.loc[fault_table.shared_field,'CSV_detected'].all()
assert fault_table.RDF_export_detected.all()

## 5. Limited I/O benchmark (not end-to-end latency)
Re-reading both files from the same archive or directory measures **CSV read times**, not the RDF/SHACL/SPARQL workflow. Source input is ~49MB uncompressed; RDF export is ~80MB uncompressed; differences in file schema and representation limit timing comparability. Report them only as supplementary engineering observations, **not as a semantic advantage or speedup**.

In [ ]:
timing=[]
for trial in range(3):
    for pipeline,path in [('CSV raw',RAW),('RDF/SPARQL export (precomputed)',SEM)]:
        t=time.perf_counter();d=read_source(path);elapsed=time.perf_counter()-t
        timing.append({'trial':trial+1,'artifact':pipeline,'seconds_to_load_precomputed_csv':elapsed,'rows':len(d)})
timing_df=pd.DataFrame(timing)
display(timing_df.groupby('artifact').agg(median_read_seconds=('seconds_to_load_precomputed_csv','median'),min_read_seconds=('seconds_to_load_precomputed_csv','min'),max_read_seconds=('seconds_to_load_precomputed_csv','max')))
timing_df.to_csv(OUTPUT_DIR/'06_precomputed_csv_read_times.csv',index=False)

## 6. Interpretation + reproducibility manifest
**What can be concluded:** Both exports retain all original measurement keys with numerical values equal within `1e-5`; the semantic extraction also exposes queryable point/observation identifiers which are absent from the raw CSV export; both conventional and semantic-aware table validators detect the five shared faults when given equivalent validation rules. **What cannot be concluded:** an intrinsic data-quality superiority, a preparation-time benefit, an IFC-to-UUID mapping, an end-to-end provenance guarantee, or any increase in forecasting accuracy.

For a truly independent RDF versus conventional workflow trial, rerun two **fresh complete pipelines** on the same raw input with predefined equivalent checks, capturing total execution time, human effort and SHACL validation results, not merely the CSV exports.

In [ ]:
manifest={
 'experiment':'E5 artifact-level controlled comparison of raw and semantic-exported observations',
 'scope':'source-to-RDF export reconciliation, metadata inventory, simulated field faults, precomputed CSV read timings',
 'NOT_evaluated':['end-to-end RDF conversion/query execution time','SHACL runtime','manual preparation effort','forecasting RMSE changes','independent cross-system interoperability'],
 'source':str(SOURCE.resolve()),'raw_path':RAW,'semantic_export_path':SEM,
 'hash_raw_sha256':bytes_hash(RAW),'hash_rdf_export_sha256':bytes_hash(SEM),
 'key_comparison':'(uuid, exact UTC timestamp, variable) one-to-one',
 'numeric_tolerance_abs':1e-5,
 'reconciliation':reconcile,'semantic_checks':semantic_checks,
 'fault_protocol':'seven synthetic single-field faults on copies; conventional CSV validator given equivalent unit/variable dictionaries',
 'python':platform.python_version(),'pandas':pd.__version__,
 'interpretation':'Descriptive artifact audit. No causal RDF benefit established.'
}
(OUTPUT_DIR/'07_E5_protocol_and_limits.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Saved E5 results to',OUTPUT_DIR.resolve())